In [1]:
# Cell 1 — Configuration and imports

from pathlib import Path
import re
import warnings

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import umap

# --- paths ---
DATA_ROOT       = Path.home() / "vambersky_t/Data"
WINDOWS_DIR     = DATA_ROOT / "extracted_windows"
EMBEDDINGS_DIR  = DATA_ROOT / "embeddings"
EXPLORATION_DIR = DATA_ROOT / "exploration"
CACHE_DIR       = EXPLORATION_DIR / "cache"
FIGURES_DIR     = EXPLORATION_DIR / "figures"

for d in (EXPLORATION_DIR, CACHE_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- embedding geometry (must match 02_embeddings) ---
SEQ_LEN   = 10_000
BIN_SIZE  = 50
N_BINS    = SEQ_LEN // BIN_SIZE   # 200
EMBED_DIM = 4096

# --- reproducibility ---
RANDOM_SEED = 42

# --- TFs in scope ---
TFS_TO_PROCESS = ["MYC", "CTCF"]

# --- plot style ---
sns.set_theme(context="paper", style="whitegrid", font_scale=1.1)
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.dpi"]  = 120
plt.rcParams["svg.fonttype"] = "none"   # keep text as text in SVG

# --- sanity ---
assert DATA_ROOT.exists(),      f"DATA_ROOT missing: {DATA_ROOT}"
assert EMBEDDINGS_DIR.exists(), f"EMBEDDINGS_DIR missing: {EMBEDDINGS_DIR}"
assert WINDOWS_DIR.exists(),    f"WINDOWS_DIR missing: {WINDOWS_DIR}"

print(f"DATA_ROOT       : {DATA_ROOT}")
print(f"EMBEDDINGS_DIR  : {EMBEDDINGS_DIR}")
print(f"CACHE_DIR       : {CACHE_DIR}")
print(f"FIGURES_DIR     : {FIGURES_DIR}")
print(f"Geometry        : {N_BINS} bins × {BIN_SIZE} bp = {SEQ_LEN} bp, embed_dim={EMBED_DIM}")

/home/jovyan/envs/exploration/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_ROOT       : /home/jovyan/vambersky_t/Data
EMBEDDINGS_DIR  : /home/jovyan/vambersky_t/Data/embeddings
CACHE_DIR       : /home/jovyan/vambersky_t/Data/exploration/cache
FIGURES_DIR     : /home/jovyan/vambersky_t/Data/exploration/figures
Geometry        : 200 bins × 50 bp = 10000 bp, embed_dim=4096


In [2]:
# file discovery and inventory

def parse_accession_tf_biosample(stem: str) -> tuple[str, str, str]:
    """
    Parse 'ENCFF765CKK__MYC__GM12878' into (accession, tf, biosample).
    Raises ValueError if the stem doesn't match the expected format.
    """
    parts = stem.split("__")
    if len(parts) != 3:
        raise ValueError(f"Unexpected filename format: {stem!r}")
    return parts[0], parts[1], parts[2]


def discover_embedding_files(tfs: list[str]) -> pl.DataFrame:
    """
    Walk EMBEDDINGS_DIR for each TF in `tfs`, pair .npz with .parquet sidecars,
    parse metadata from filenames, and check that the corresponding windows CSV
    exists. Returns an inventory DataFrame with one row per usable file.
    """
    rows = []
    for tf in tfs:
        tf_dir = EMBEDDINGS_DIR / tf
        if not tf_dir.exists():
            warnings.warn(f"No embedding directory for TF '{tf}': {tf_dir}")
            continue

        for npz_path in sorted(tf_dir.glob("*.embeddings.npz")):
            if ".checkpoint" in npz_path.name:
                continue
            # derive matching sidecar and windows paths
            base = npz_path.name.removesuffix(".embeddings.npz")
            parquet_path = tf_dir / f"{base}.peak_ids.parquet"
            windows_path = WINDOWS_DIR / tf / f"{base}.full_peaks.csv.gz"

            try:
                accession, parsed_tf, biosample = parse_accession_tf_biosample(base)
            except ValueError as e:
                warnings.warn(f"Skipping {npz_path.name}: {e}")
                continue

            # consistency checks
            if parsed_tf != tf:
                warnings.warn(
                    f"Skipping {npz_path.name}: filename TF '{parsed_tf}' "
                    f"doesn't match directory TF '{tf}'"
                )
                continue
            if not parquet_path.exists():
                warnings.warn(
                    f"Skipping {npz_path.name}: missing sidecar {parquet_path.name}"
                )
                continue
            if not windows_path.exists():
                warnings.warn(
                    f"Skipping {npz_path.name}: missing windows CSV {windows_path.name}"
                )
                continue

            rows.append({
                "accession":    accession,
                "tf":           tf,
                "biosample":    biosample,
                "base":         base,
                "npz_path":     str(npz_path),
                "parquet_path": str(parquet_path),
                "windows_path": str(windows_path),
                "npz_bytes":    npz_path.stat().st_size,
            })

    if not rows:
        raise RuntimeError(
            f"No usable embedding files found under {EMBEDDINGS_DIR} for TFs {tfs}"
        )

    return pl.DataFrame(rows).sort(["tf", "biosample", "accession"])


inventory = discover_embedding_files(TFS_TO_PROCESS)

# summary
print(f"Usable files: {len(inventory)}")
print(f"Total compressed size: {inventory['npz_bytes'].sum() / 1e9:.2f} GB")
print()
print("By TF:")
print(inventory.group_by("tf").len().sort("tf"))
print()
print("Full inventory:")
print(inventory.select(["tf", "biosample", "accession", "npz_bytes"]))

Usable files: 3
Total compressed size: 21.55 GB

By TF:
shape: (2, 2)
┌──────┬─────┐
│ tf   ┆ len │
│ ---  ┆ --- │
│ str  ┆ u32 │
╞══════╪═════╡
│ CTCF ┆ 1   │
│ MYC  ┆ 2   │
└──────┴─────┘

Full inventory:
shape: (3, 4)
┌──────┬───────────┬─────────────┬─────────────┐
│ tf   ┆ biosample ┆ accession   ┆ npz_bytes   │
│ ---  ┆ ---       ┆ ---         ┆ ---         │
│ str  ┆ str       ┆ str         ┆ i64         │
╞══════╪═══════════╪═════════════╪═════════════╡
│ CTCF ┆ GM12878   ┆ ENCFF017XLW ┆ 12529411526 │
│ MYC  ┆ GM12878   ┆ ENCFF765CKK ┆ 2786686102  │
│ MYC  ┆ H1        ┆ ENCFF700CXD ┆ 6234904618  │
└──────┴───────────┴─────────────┴─────────────┘


In [3]:
# Cell 3 — Helper functions

def compute_gc_profile(
    sequences: list[str],
    n_bins: int = N_BINS,
    bin_size: int = BIN_SIZE,
) -> np.ndarray:
    """
    Compute per-bin GC fraction for a list of sequences.

    GC is computed as (G + C) / (A + C + G + T) — Ns are excluded from
    both numerator and denominator. A bin that is entirely N yields NaN.

    Returns
    -------
    gc : np.ndarray, shape (len(sequences), n_bins), dtype float32
        GC fraction per bin. NaN where a bin has no non-N bases.
    """
    n = len(sequences)
    seq_len = n_bins * bin_size
    gc = np.empty((n, n_bins), dtype=np.float32)

    for i, seq in enumerate(sequences):
        if len(seq) != seq_len:
            raise ValueError(
                f"Sequence {i} has length {len(seq)}, expected {seq_len}"
            )

        arr = np.frombuffer(seq.encode("ascii"), dtype=np.uint8)  # (seq_len,)
        arr = arr.reshape(n_bins, bin_size)                       # (n_bins, bin_size)

        is_g = (arr == ord("G"))
        is_c = (arr == ord("C"))
        is_a = (arr == ord("A"))
        is_t = (arr == ord("T"))

        gc_count   = (is_g | is_c).sum(axis=1).astype(np.float32)
        acgt_count = (is_a | is_c | is_g | is_t).sum(axis=1).astype(np.float32)

        with np.errstate(divide="ignore", invalid="ignore"):
            gc[i] = np.where(acgt_count > 0, gc_count / acgt_count, np.nan)

    return gc


def load_peak_meta_and_sequences(
    parquet_path: Path,
    windows_path: Path,
) -> pl.DataFrame:
    """
    Load the sidecar parquet (authoritative ordering for the embedding array)
    and attach sequences from the windows CSV via left join on peak_id.

    Returns a DataFrame with columns [peak_id, chr, start, end, center, sequence]
    in the order that matches the embedding array rows.
    """
    sidecar = pl.read_parquet(parquet_path)  # peak_id, chr, start, end, center
    windows = pl.read_csv(windows_path).select(["peak_id", "sequence"])

    merged = sidecar.join(windows, on="peak_id", how="left")

    missing = merged.filter(pl.col("sequence").is_null()).height
    if missing > 0:
        raise RuntimeError(
            f"{missing} peak_ids from {parquet_path.name} have no matching "
            f"sequence in {windows_path.name}. Data integrity failure."
        )

    # Preserve sidecar row order (join in polars should respect left order,
    # but be explicit: reassert by row index).
    assert merged.height == sidecar.height, "Row count changed after join"

    return merged


def align_and_drop_nan(
    embeddings: np.ndarray,
    peak_meta: pl.DataFrame,
    gc: np.ndarray,
) -> tuple[np.ndarray, pl.DataFrame, np.ndarray, int]:
    """
    Drop peaks where any 50 bp bin is entirely N-masked (NaN in gc).

    Assumes embeddings, peak_meta, and gc are already in the same row order —
    which holds by construction: the sidecar parquet defines the order, and
    peak_meta and gc are derived from it.

    Returns
    -------
    embeddings_clean : (n_kept, N_BINS, EMBED_DIM) float16
    peak_meta_clean  : polars DataFrame with n_kept rows
    gc_clean         : (n_kept, N_BINS) float32
    n_dropped        : int
    """
    assert embeddings.shape[0] == peak_meta.height == gc.shape[0], (
        f"Row count mismatch: embeddings={embeddings.shape[0]}, "
        f"peak_meta={peak_meta.height}, gc={gc.shape[0]}"
    )

    keep_mask = ~np.isnan(gc).any(axis=1)   # True where all bins have data
    n_dropped = int((~keep_mask).sum())

    embeddings_clean = embeddings[keep_mask]
    gc_clean         = gc[keep_mask]
    peak_meta_clean  = peak_meta.filter(pl.Series(keep_mask))

    return embeddings_clean, peak_meta_clean, gc_clean, n_dropped


# --- quick smoke test on a synthetic sequence ---
_test_seq = ("ACGT" * 2500)  # 10 kb, 50% GC, no Ns
_test_gc = compute_gc_profile([_test_seq])
assert _test_gc.shape == (1, N_BINS)
assert np.allclose(_test_gc, 0.5), f"Expected 0.5 GC, got {_test_gc.mean()}"

_test_seq_n = "N" * 10_000  # all Ns → all NaN
_test_gc_n = compute_gc_profile([_test_seq_n])
assert np.isnan(_test_gc_n).all()

print("Helper functions defined. Smoke tests passed.")

Helper functions defined. Smoke tests passed.


In [4]:
row = inventory.row(0, named=True)
meta = load_peak_meta_and_sequences(Path(row["parquet_path"]), Path(row["windows_path"]))
print(meta.shape, meta.columns)
gc_test = compute_gc_profile(meta["sequence"].to_list()[:5])
print(gc_test.shape, gc_test.mean(), np.isnan(gc_test).sum())

(9966, 6) ['peak_id', 'chr', 'start', 'end', 'center', 'sequence']
(5, 200) 0.45890003 0


In [5]:
# Cell 4 — Main processing loop

import gc as gc_module
import time


def cache_paths_for(base: str, tf: str) -> tuple[Path, Path]:
    """Derive cache paths for a given file's derived outputs."""
    out_dir = CACHE_DIR / tf
    out_dir.mkdir(parents=True, exist_ok=True)
    npz     = out_dir / f"{base}.derived.npz"
    parquet = out_dir / f"{base}.peak_meta.parquet"
    return npz, parquet


def process_one_file(row: dict) -> dict:
    """
    Process a single embedding file: load raw array, compute derived quantities,
    cache them to disk, release the raw array, and return summary stats.

    Skip-if-exists: if both cache files already exist, return their stats
    without touching the raw npz.
    """
    base       = row["base"]
    tf         = row["tf"]
    biosample  = row["biosample"]
    accession  = row["accession"]

    cache_npz, cache_parquet = cache_paths_for(base, tf)

    if cache_npz.exists() and cache_parquet.exists():
        # cheap stat from cache without touching raw embeddings
        cached = np.load(cache_npz)
        n_kept = cached["mean_pool"].shape[0]
        return {
            "base": base, "tf": tf, "biosample": biosample, "accession": accession,
            "n_kept": n_kept, "n_dropped": int(cached["n_dropped"]),
            "elapsed_s": 0.0, "from_cache": True,
        }

    t0 = time.time()

    # 1. metadata + sequences (small)
    peak_meta = load_peak_meta_and_sequences(
        Path(row["parquet_path"]), Path(row["windows_path"]),
    )
    n_loaded = peak_meta.height

    # 2. GC (small)
    gc_arr = compute_gc_profile(peak_meta["sequence"].to_list())

    # 3. raw embeddings (LARGE — held only for this scope)
    with np.load(row["npz_path"]) as npz:
        embeddings = npz["embeddings"]   # (n_peaks, 200, 4096) float16
        # materialize from npz lazy view to a real array so the file handle can close
        embeddings = np.asarray(embeddings)

    # 4. align + drop NaN peaks
    embeddings, peak_meta, gc_arr, n_dropped = align_and_drop_nan(
        embeddings, peak_meta, gc_arr,
    )
    n_kept = embeddings.shape[0]

    # 5. derived quantities
    # mean pool over the 200-bin axis -> (n_kept, EMBED_DIM)
    # cast to float32 for the reduction to avoid float16 overflow on a sum of 200,
    # then back to float16 for storage
    mean_pool = np.empty((n_kept, EMBED_DIM), dtype=np.float16)
    chunk = 256
    for s in range(0, n_kept, chunk):
        e = min(s + chunk, n_kept)
        mean_pool[s:e] = embeddings[s:e].astype(np.float32).mean(axis=1).astype(np.float16)
    # positional profile: mean over peaks -> (N_BINS, EMBED_DIM)
    # float32 to preserve precision; this is small anyway
    pos_profile = np.zeros(embeddings.shape[1:], dtype=np.float32)  # (200, 4096)
    for s in range(0, n_kept, chunk):
        e = min(s + chunk, n_kept)
        pos_profile += embeddings[s:e].astype(np.float32).sum(axis=0)
    pos_profile /= n_kept
    # per-bin mean GC -> (N_BINS,)
    gc_mean_profile = gc_arr.mean(axis=0)

    # 6. release raw array before caching writes
    del embeddings
    gc_module.collect()

    # 7. cache
    np.savez_compressed(
        cache_npz,
        mean_pool       = mean_pool,        # (n_kept, EMBED_DIM) float16
        pos_profile     = pos_profile,      # (N_BINS, EMBED_DIM) float32
        gc              = gc_arr,           # (n_kept, N_BINS)    float32
        gc_mean_profile = gc_mean_profile,  # (N_BINS,)           float32
        n_dropped       = np.array(n_dropped, dtype=np.int32),
    )
    peak_meta.write_parquet(cache_parquet)

    elapsed = time.time() - t0

    return {
        "base": base, "tf": tf, "biosample": biosample, "accession": accession,
        "n_loaded": n_loaded, "n_kept": n_kept, "n_dropped": n_dropped,
        "elapsed_s": elapsed, "from_cache": False,
    }


# --- run loop ---
results = []
total_t0 = time.time()

for i, row in enumerate(inventory.iter_rows(named=True), 1):
    print(f"\n[{i}/{len(inventory)}] {row['tf']} / {row['biosample']} / {row['accession']}")
    stats = process_one_file(row)
    results.append(stats)

    if stats["from_cache"]:
        print(f"  cached: {stats['n_kept']} peaks kept, {stats['n_dropped']} dropped")
    else:
        print(f"  loaded {stats['n_loaded']} → kept {stats['n_kept']} "
              f"(dropped {stats['n_dropped']}) in {stats['elapsed_s']:.1f} s")

total_elapsed = time.time() - total_t0
print(f"\nLoop done in {total_elapsed:.1f} s ({total_elapsed/60:.1f} min)")

# summary table
results_df = pl.DataFrame(results)
print()
print(results_df.select(["tf", "biosample", "accession", "n_kept", "n_dropped", "elapsed_s", "from_cache"]))


[1/3] CTCF / GM12878 / ENCFF017XLW
  cached: 9922 peaks kept, 44 dropped

[2/3] MYC / GM12878 / ENCFF765CKK
  cached: 2203 peaks kept, 13 dropped

[3/3] MYC / H1 / ENCFF700CXD
  loaded 4957 → kept 4936 (dropped 21) in 444.2 s

Loop done in 447.4 s (7.5 min)

shape: (3, 7)
┌──────┬───────────┬─────────────┬────────┬───────────┬────────────┬────────────┐
│ tf   ┆ biosample ┆ accession   ┆ n_kept ┆ n_dropped ┆ elapsed_s  ┆ from_cache │
│ ---  ┆ ---       ┆ ---         ┆ ---    ┆ ---       ┆ ---        ┆ ---        │
│ str  ┆ str       ┆ str         ┆ i64    ┆ i64       ┆ f64        ┆ bool       │
╞══════╪═══════════╪═════════════╪════════╪═══════════╪════════════╪════════════╡
│ CTCF ┆ GM12878   ┆ ENCFF017XLW ┆ 9922   ┆ 44        ┆ 0.0        ┆ true       │
│ MYC  ┆ GM12878   ┆ ENCFF765CKK ┆ 2203   ┆ 13        ┆ 0.0        ┆ true       │
│ MYC  ┆ H1        ┆ ENCFF700CXD ┆ 4936   ┆ 21        ┆ 444.172906 ┆ false      │
└──────┴───────────┴─────────────┴────────┴───────────┴────────────┴──

In [6]:
# Cell 5 — Consolidate cached outputs into in-memory structures

# per-file dicts, keyed by `base` (e.g. "ENCFF017XLW__CTCF__GM12878")
mean_pool_by_file       = {}   # base -> (n_kept, EMBED_DIM) float16
pos_profile_by_file     = {}   # base -> (N_BINS, EMBED_DIM) float32
gc_by_file              = {}   # base -> (n_kept, N_BINS) float32
gc_mean_profile_by_file = {}   # base -> (N_BINS,) float32
meta_by_file            = {}   # base -> polars DataFrame (n_kept rows)

for row in inventory.iter_rows(named=True):
    base = row["base"]
    cache_npz, cache_parquet = cache_paths_for(base, row["tf"])

    if not (cache_npz.exists() and cache_parquet.exists()):
        warnings.warn(f"Skipping {base}: cache missing (cell 4 incomplete?)")
        continue

    cached = np.load(cache_npz)
    mean_pool_by_file[base]       = cached["mean_pool"]
    pos_profile_by_file[base]     = cached["pos_profile"]
    gc_by_file[base]              = cached["gc"]
    gc_mean_profile_by_file[base] = cached["gc_mean_profile"]

    meta = pl.read_parquet(cache_parquet)
    # tag with file-level metadata so we can group by these later
    meta = meta.with_columns([
        pl.lit(base).alias("file_base"),
        pl.lit(row["tf"]).alias("tf"),
        pl.lit(row["biosample"]).alias("biosample"),
        pl.lit(row["accession"]).alias("accession"),
    ])
    meta_by_file[base] = meta


# --- stacked structures for Block 1 (global PCA/UMAP) ---

# concatenate mean pools across all files -> (n_total_kept, EMBED_DIM)
all_bases       = list(mean_pool_by_file.keys())
mean_pool_all   = np.vstack([mean_pool_by_file[b] for b in all_bases])
meta_polars_all = pl.concat(
    [meta_by_file[b] for b in all_bases],
    how="vertical",
)

# sanity check: row counts match
assert mean_pool_all.shape[0] == meta_polars_all.height, (
    f"Stacked mean pool ({mean_pool_all.shape[0]}) and metadata "
    f"({meta_polars_all.height}) disagree"
)

# --- summary ---
print(f"Files loaded: {len(mean_pool_by_file)}")
print(f"Total peaks (kept): {mean_pool_all.shape[0]}")
print(f"mean_pool_all      : {mean_pool_all.shape} {mean_pool_all.dtype}  "
      f"({mean_pool_all.nbytes / 1e6:.1f} MB)")
print()
print("Per-file shapes:")
for b in all_bases:
    print(f"  {b}")
    print(f"    mean_pool   : {mean_pool_by_file[b].shape}")
    print(f"    pos_profile : {pos_profile_by_file[b].shape}")
    print(f"    gc          : {gc_by_file[b].shape}")
print()
print("Per-TF peak counts:")
print(meta_polars_all.group_by("tf").len().sort("tf"))
print()
print("Per-(TF, biosample) peak counts:")
print(
    meta_polars_all
    .group_by(["tf", "biosample"]).len()
    .sort(["tf", "biosample"])
)

Files loaded: 3
Total peaks (kept): 17061
mean_pool_all      : (17061, 4096) float16  (139.8 MB)

Per-file shapes:
  ENCFF017XLW__CTCF__GM12878
    mean_pool   : (9922, 4096)
    pos_profile : (200, 4096)
    gc          : (9922, 200)
  ENCFF765CKK__MYC__GM12878
    mean_pool   : (2203, 4096)
    pos_profile : (200, 4096)
    gc          : (2203, 200)
  ENCFF700CXD__MYC__H1
    mean_pool   : (4936, 4096)
    pos_profile : (200, 4096)
    gc          : (4936, 200)

Per-TF peak counts:
shape: (2, 2)
┌──────┬──────┐
│ tf   ┆ len  │
│ ---  ┆ ---  │
│ str  ┆ u32  │
╞══════╪══════╡
│ CTCF ┆ 9922 │
│ MYC  ┆ 7139 │
└──────┴──────┘

Per-(TF, biosample) peak counts:
shape: (3, 3)
┌──────┬───────────┬──────┐
│ tf   ┆ biosample ┆ len  │
│ ---  ┆ ---       ┆ ---  │
│ str  ┆ str       ┆ u32  │
╞══════╪═══════════╪══════╡
│ CTCF ┆ GM12878   ┆ 9922 │
│ MYC  ┆ GM12878   ┆ 2203 │
│ MYC  ┆ H1        ┆ 4936 │
└──────┴───────────┴──────┘


at this point the data is in memory, next step is the PCA. once on unbalanced, then on subsample.